# 第3章　環境ゼロで始める ― Google Colab と Jupyter Notebook

**『ゼロから動かす医療診断支援AI（入門編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-intro

## 3.2　最初の一歩 ― ノートブックとGPUランタイム

In [ ]:
import torch
print(torch.cuda.is_available())          # True と出れば成功
print(torch.cuda.get_device_name(0))      # 割り当てられたGPUの名前（例: Tesla T4）

```text
!nvidia-smi
```

## 3.3　ライブラリのインストール

```text
!pip install pydicom nibabel monai
```

## 3.4　Googleドライブをマウントする ― データの置き場所

In [ ]:
from google.colab import drive
drive.mount('/content/drive')             # 認証を求められる → 許可する
# 以後 /content/drive/MyDrive/ が自分のドライブとして使える
import os
os.listdir('/content/drive/MyDrive')      # まずは直下を一覧（dataset等は自分で作る）

## 3.5　データのアップロードと、医療データの鉄則

In [ ]:
from google.colab import files
uploaded = files.upload()                 # ダイアログでファイルを選ぶ

## 3.7　Colabで小さく学習を回してみる

In [ ]:
import torch.nn as nn
import torch, torch.nn as nn
device = "cuda" if torch.cuda.is_available() else "cpu"

model = nn.Sequential(nn.Flatten(), nn.Linear(28*28, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss()

for images, labels in train_loader:                 # DataLoaderは基礎編で学ぶ
    images, labels = images.to(device), labels.to(device)   # データもGPUへ
    opt.zero_grad()
    loss = lossf(model(images), labels)
    loss.backward(); opt.step()

## 3.8　同じ体験をローカルでも ― Jupyter Notebook と JupyterLab

```bash
pip install jupyterlab       # 一度だけインストール
jupyter lab                  # 起動 → ブラウザが開き、ノートブックが使える
```

## 3.10　公開データをColabに引き込む ― Kaggle・Hugging Face・wget・gdown

```text
import os
from google.colab import userdata            # Colabのシークレットを読む（鍵は表に出さない）
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")

!pip -q install kaggle
# 例：眼底（糖尿病網膜症の有無）の公開データを /content に取得して展開する
!kaggle datasets download -d pkdarabi/diagnosis-of-diabetic-retinopathy -p /content
!unzip -q /content/diagnosis-of-diabetic-retinopathy.zip -d /content/dr
```

```text
!pip -q install huggingface_hub
from huggingface_hub import snapshot_download
# repo_type="dataset" を忘れない（既定はモデル置き場を見に行ってしまう）
local = snapshot_download(repo_id="<公開データセット名>", repo_type="dataset",
                          local_dir="/content/hf_data")
import os; print(os.listdir(local))          # 落ちてきた中身を確認
```

```text
# ① 素のURLから直接ダウンロード
!wget -q "https://example.org/dataset.zip" -O /content/dataset.zip

# ② Googleドライブの「リンクを知っている全員が閲覧可」の共有リンクから
!pip -q install -U gdown
!gdown --fuzzy "https://drive.google.com/file/d/XXXXXXXXXXXX/view?usp=sharing" -O /content/data.zip
```

```text
!unzip -q /content/data.zip -d /content/data      # .zip
# !tar xf /content/data.tar.gz -C /content/data   # .tar.gz のとき
# フォルダの形を2階層だけ覗く（画像がどう並んでいるか）
!find /content/data -maxdepth 2 -type d | head
```

```text
import os, shutil
CACHE = "/content/drive/MyDrive/datasets/dr.zip"    # ドライブ側の退避先
LOCAL = "/content/diagnosis-of-diabetic-retinopathy.zip"  # kaggleが落とすファイル名に合わせる
if os.path.exists(CACHE):
    shutil.copy(CACHE, LOCAL)                        # 2回目以降：ドライブから瞬時にコピー
else:
    !kaggle datasets download -d pkdarabi/diagnosis-of-diabetic-retinopathy -p /content -o
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)   # 退避先の親フォルダを先に作る
    shutil.copy(LOCAL, CACHE)                        # 初回だけ：落として退避しておく
!unzip -q -o {LOCAL} -d /content/dr    # {LOCAL} は直前で定義したPython変数の中身が埋め込まれる
```

## 3.11　データを触る前に一覧で眺める ― サムネイルのコンタクトシート

In [ ]:
import matplotlib.pyplot as plt
from torchvision import datasets
import random

ds = datasets.ImageFolder("/content/dr/train")   # (PIL画像, ラベル番号)
print("クラス対応:", ds.class_to_idx)            # 例 {'DR':0,'No_DR':1}

# 各クラスから4枚ずつ、ラベルを付けて並べる
per_class = 4
by_cls = {c: [] for c in ds.class_to_idx.values()}
for idx in random.sample(range(len(ds)), len(ds)):
    _, y = ds.samples[idx]
    if len(by_cls[y]) < per_class:
        by_cls[y].append(idx)
    if all(len(v) == per_class for v in by_cls.values()):
        break

names = {v: k for k, v in ds.class_to_idx.items()}
picks = [i for cls in by_cls for i in by_cls[cls]]
plt.figure(figsize=(12, 6))
for k, i in enumerate(picks):
    img, y = ds[i]
    plt.subplot(len(by_cls), per_class, k + 1)
    plt.imshow(img); plt.axis("off")               # 眼底はカラー(3ch)なので cmap は指定しない
    plt.title(names[y], fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
counts = np.bincount([y for _, y in ds.samples])
plt.bar([names[i] for i in range(len(counts))], counts)
plt.ylabel("count"); plt.title("images per class"); plt.show()   # 図中は英語（日本語は□に化ける）
print(dict(zip([names[i] for i in range(len(counts))], counts.tolist())))

## 3.12　学習曲線をその場で描く ― ライブ更新プロット

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

hist = {"train_loss": [], "val_auc": []}

def live_plot(hist):
    clear_output(wait=True)                       # 前回の図を消してから描き直す
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(hist["train_loss"], marker="o"); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
    ax[1].plot(hist["val_auc"],  marker="o", color="tab:green")
    ax[1].set_title("val AUC"); ax[1].set_xlabel("epoch"); ax[1].set_ylim(0.4, 1.0)
    ax[1].axhline(0.5, ls="--", lw=0.8, color="gray")   # 0.5＝でたらめ。下回ったらラベル取り違えを疑う
    plt.tight_layout(); plt.show()

# 学習ループの中で、1エポックごとに記録して描く（loss計算・検証AUCは各章のとおり）
for epoch in range(EPOCHS):        # EPOCHS・epoch_loss・epoch_auc は、あなたの学習ループで定義済みの変数に読み替えてください
    # ... 学習1周（train_loss を集計）...
    # ... 検証で val_auc を計算 ...
    hist["train_loss"].append(epoch_loss)
    hist["val_auc"].append(epoch_auc)
    live_plot(hist)

## 3.13　結果を保存して手元にダウンロードする ― CSV・図・重み

In [ ]:
import pandas as pd
# 検証・テストのループで集めた ids（症例名）, probs（陽性確率）, trues（正解）から表を作る
# ※ ids/probs/trues は、あなたの評価ループで作った変数に読み替えてください（この3行だけでは動きません）
res = pd.DataFrame({"id": ids, "prob": probs, "true": trues})
res["pred"] = (res["prob"] >= 0.5).astype(int)
res = res.sort_values("prob", ascending=False)     # 確率の高い順（陽性らしい順）に
res.to_csv("/content/predictions.csv", index=False)

from google.colab import files
files.download("/content/predictions.csv")          # ブラウザ経由で手元PCへ保存

```text
import matplotlib.pyplot as plt, os
os.makedirs("/content/outputs", exist_ok=True)
# ※ plt.show() を済ませた図は閉じられているので、あとから plt.savefig を呼んでも
#    白紙のPNGが保存される。図は「描いたその場で」fig を受けて保存すること。
#    （live_plot の中で fig.savefig(...) を呼ぶのが確実）
res.to_csv("/content/outputs/predictions.csv", index=False)
torch.save(model.state_dict(), "/content/outputs/model.pth")   # 重みも成果物

!zip -qr /content/outputs.zip /content/outputs      # まとめて圧縮
from google.colab import files
files.download("/content/outputs.zip")              # 図・CSV・重みを一括ダウンロード
```

## 3.14　つまずき回避・上級編 ― 実行順・メモリ・秘密情報

In [ ]:
imgs = [Image.open(p).convert("RGB") for p in all_paths]   # 数万枚だとRAMを食い潰して落ちる

In [ ]:
import gc, torch
del model, opt                    # もう使わないモデル・オプティマイザを捨てる
gc.collect(); torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() // 1024**2, "MB 使用中")   # 減っていれば成功